# Subject: c_analyzer
Source: /home/devuser/edu_data/datasets/TrainingDB/EncryT/cpython/Tools/c-analyzer/c_analyzer

### File: __main__.py

In [ ]:
import io
import logging
import os
import os.path
import re
import sys

from c_common import fsutil
from c_common.logging import VERBOSITY, Printer
from c_common.scriptutil import (
    add_verbosity_cli,
    add_traceback_cli,
    add_sepval_cli,
    add_progress_cli,
    add_files_cli,
    add_commands_cli,
    process_args_by_key,
    configure_logger,
    get_prog,
    filter_filenames,
)
from c_parser.info import KIND
from .match import filter_forward
from . import (
    analyze as _analyze,
    datafiles as _datafiles,
    check_all as _check_all,
)


KINDS = [
    KIND.TYPEDEF,
    KIND.STRUCT,
    KIND.UNION,
    KIND.ENUM,
    KIND.FUNCTION,
    KIND.VARIABLE,
    KIND.STATEMENT,
]

logger = logging.getLogger(__name__)


#######################################
# table helpers

TABLE_SECTIONS = {
    'types': (
        ['kind', 'name', 'data', 'file'],
        KIND.is_type_decl,
        (lambda v: (v.kind.value, v.filename or '', v.name)),
    ),
    'typedefs': 'types',
    'structs': 'types',
    'unions': 'types',
    'enums': 'types',
    'functions': (
        ['name', 'data', 'file'],
        (lambda kind: kind is KIND.FUNCTION),
        (lambda v: (v.filename or '', v.name)),
    ),
    'variables': (
        ['name', 'parent', 'data', 'file'],
        (lambda kind: kind is KIND.VARIABLE),
        (lambda v: (v.filename or '', str(v.parent) if v.parent else '', v.name)),
    ),
    'statements': (
        ['file', 'parent', 'data'],
        (lambda kind: kind is KIND.STATEMENT),
        (lambda v: (v.filename or '', str(v.parent) if v.parent else '', v.name)),
    ),
    KIND.TYPEDEF: 'typedefs',
    KIND.STRUCT: 'structs',
    KIND.UNION: 'unions',
    KIND.ENUM: 'enums',
    KIND.FUNCTION: 'functions',
    KIND.VARIABLE: 'variables',
    KIND.STATEMENT: 'statements',
}


def _render_table(items, columns, relroot=None):
    # XXX improve this
    header = '\t'.join(columns)
    div = '--------------------'
    yield header
    yield div
    total = 0
    for item in items:
        rowdata = item.render_rowdata(columns)
        row = [rowdata[c] for c in columns]
        if relroot and 'file' in columns:
            index = columns.index('file')
            row[index] = os.path.relpath(row[index], relroot)
        yield '\t'.join(row)
        total += 1
    yield div
    yield f'total: {total}'


def build_section(name, groupitems, *, relroot=None):
    info = TABLE_SECTIONS[name]
    while type(info) is not tuple:
        if name in KINDS:
            name = info
        info = TABLE_SECTIONS[info]

    columns, match_kind, sortkey = info
    items = (v for v in groupitems if match_kind(v.kind))
    items = sorted(items, key=sortkey)
    def render():
        yield ''
        yield f'{name}:'
        yield ''
        for line in _render_table(items, columns, relroot):
            yield line
    return items, render


#######################################
# the checks

CHECKS = {
    #'globals': _check_globals,
}


def add_checks_cli(parser, checks=None, *, add_flags=None):
    default = False
    if not checks:
        checks = list(CHECKS)
        default = True
    elif isinstance(checks, str):
        checks = [checks]
    if (add_flags is None and len(checks) > 1) or default:
        add_flags = True

    process_checks = add_sepval_cli(parser, '--check', 'checks', checks)
    if add_flags:
        for check in checks:
            parser.add_argument(f'--{check}', dest='checks',
                                action='append_const', const=check)
    return [
        process_checks,
    ]


def _get_check_handlers(fmt, printer, verbosity=VERBOSITY):
    div = None
    def handle_after():
        pass
    if not fmt:
        div = ''
        def handle_failure(failure, data):
            data = repr(data)
            if verbosity >= 3:
                logger.info(f'failure: {failure}')
                logger.info(f'data:    {data}')
            else:
                logger.warn(f'failure: {failure} (data: {data})')
    elif fmt == 'raw':
        def handle_failure(failure, data):
            print(f'{failure!r} {data!r}')
    elif fmt == 'brief':
        def handle_failure(failure, data):
            parent = data.parent or ''
            funcname = parent if isinstance(parent, str) else parent.name
            name = f'({funcname}).{data.name}' if funcname else data.name
            failure = failure.split('\t')[0]
            print(f'{data.filename}:{name} - {failure}')
    elif fmt == 'summary':
        def handle_failure(failure, data):
            print(_fmt_one_summary(data, failure))
    elif fmt == 'full':
        div = ''
        def handle_failure(failure, data):
            name = data.shortkey if data.kind is KIND.VARIABLE else data.name
            parent = data.parent or ''
            funcname = parent if isinstance(parent, str) else parent.name
            known = 'yes' if data.is_known else '*** NO ***'
            print(f'{data.kind.value} {name!r} failed ({failure})')
            print(f'  file:         {data.filename}')
            print(f'  func:         {funcname or "-"}')
            print(f'  name:         {data.name}')
            print(f'  data:         ...')
            print(f'  type unknown: {known}')
    else:
        if fmt in FORMATS:
            raise NotImplementedError(fmt)
        raise ValueError(f'unsupported fmt {fmt!r}')
    return handle_failure, handle_after, div


#######################################
# the formats

def fmt_raw(analysis):
    for item in analysis:
        yield from item.render('raw')


def fmt_brief(analysis):
    # XXX Support sorting.
    items = sorted(analysis)
    for kind in KINDS:
        if kind is KIND.STATEMENT:
            continue
        for item in items:
            if item.kind is not kind:
                continue
            yield from item.render('brief')
    yield f'  total: {len(items)}'


def fmt_summary(analysis):
    # XXX Support sorting and grouping.
    items = list(analysis)
    total = len(items)

    def section(name):
        _, render = build_section(name, items)
        yield from render()

    yield from section('types')
    yield from section('functions')
    yield from section('variables')
    yield from section('statements')

    yield ''
#    yield f'grand total: {len(supported) + len(unsupported)}'
    yield f'grand total: {total}'


def _fmt_one_summary(item, extra=None):
    parent = item.parent or ''
    funcname = parent if isinstance(parent, str) else parent.name
    if extra:
        return f'{item.filename:35}\t{funcname or "-":35}\t{item.name:40}\t{extra}'
    else:
        return f'{item.filename:35}\t{funcname or "-":35}\t{item.name}'


def fmt_full(analysis):
    # XXX Support sorting.
    items = sorted(analysis, key=lambda v: v.key)
    yield ''
    for item in items:
        yield from item.render('full')
        yield ''
    yield f'total: {len(items)}'


FORMATS = {
    'raw': fmt_raw,
    'brief': fmt_brief,
    'summary': fmt_summary,
    'full': fmt_full,
}


def add_output_cli(parser, *, default='summary'):
    parser.add_argument('--format', dest='fmt', default=default, choices=tuple(FORMATS))

    def process_args(args, *, argv=None):
        pass
    return process_args


#######################################
# the commands

def _cli_check(parser, checks=None, **kwargs):
    if isinstance(checks, str):
        checks = [checks]
    if checks is False:
        process_checks = None
    elif checks is None:
        process_checks = add_checks_cli(parser)
    elif len(checks) == 1 and type(checks) is not dict and re.match(r'^<.*>$', checks[0]):
        check = checks[0][1:-1]
        def process_checks(args, *, argv=None):
            args.checks = [check]
    else:
        process_checks = add_checks_cli(parser, checks=checks)
    process_progress = add_progress_cli(parser)
    process_output = add_output_cli(parser, default=None)
    process_files = add_files_cli(parser, **kwargs)
    return [
        process_checks,
        process_progress,
        process_output,
        process_files,
    ]


def cmd_check(filenames, *,
              checks=None,
              ignored=None,
              fmt=None,
              failfast=False,
              iter_filenames=None,
              relroot=fsutil.USE_CWD,
              track_progress=None,
              verbosity=VERBOSITY,
              _analyze=_analyze,
              _CHECKS=CHECKS,
              **kwargs
              ):
    if not checks:
        checks = _CHECKS
    elif isinstance(checks, str):
        checks = [checks]
    checks = [_CHECKS[c] if isinstance(c, str) else c
              for c in checks]
    printer = Printer(verbosity)
    (handle_failure, handle_after, div
     ) = _get_check_handlers(fmt, printer, verbosity)

    filenames, relroot = fsutil.fix_filenames(filenames, relroot=relroot)
    filenames = filter_filenames(filenames, iter_filenames, relroot)
    if track_progress:
        filenames = track_progress(filenames)

    logger.info('analyzing files...')
    analyzed = _analyze(filenames, **kwargs)
    analyzed.fix_filenames(relroot, normalize=False)
    decls = filter_forward(analyzed, markpublic=True)

    logger.info('checking analysis results...')
    failed = []
    for data, failure in _check_all(decls, checks, failfast=failfast):
        if data is None:
            printer.info('stopping after one failure')
            break
        if div is not None and len(failed) > 0:
            printer.info(div)
        failed.append(data)
        handle_failure(failure, data)
    handle_after()

    printer.info('-------------------------')
    logger.info(f'total failures: {len(failed)}')
    logger.info('done checking')

    if fmt == 'summary':
        print('Categorized by storage:')
        print()
        from .match import group_by_storage
        grouped = group_by_storage(failed, ignore_non_match=False)
        for group, decls in grouped.items():
            print()
            print(group)
            for decl in decls:
                print(' ', _fmt_one_summary(decl))
            print(f'subtotal: {len(decls)}')

    if len(failed) > 0:
        sys.exit(len(failed))


def _cli_analyze(parser, **kwargs):
    process_progress = add_progress_cli(parser)
    process_output = add_output_cli(parser)
    process_files = add_files_cli(parser, **kwargs)
    return [
        process_progress,
        process_output,
        process_files,
    ]


# XXX Support filtering by kind.
def cmd_analyze(filenames, *,
                fmt=None,
                iter_filenames=None,
                relroot=fsutil.USE_CWD,
                track_progress=None,
                verbosity=None,
                _analyze=_analyze,
                formats=FORMATS,
                **kwargs
                ):
    verbosity = verbosity if verbosity is not None else 3

    try:
        do_fmt = formats[fmt]
    except KeyError:
        raise ValueError(f'unsupported fmt {fmt!r}')

    filenames, relroot = fsutil.fix_filenames(filenames, relroot=relroot)
    filenames = filter_filenames(filenames, iter_filenames, relroot)
    if track_progress:
        filenames = track_progress(filenames)

    logger.info('analyzing files...')
    analyzed = _analyze(filenames, **kwargs)
    analyzed.fix_filenames(relroot, normalize=False)
    decls = filter_forward(analyzed, markpublic=True)

    for line in do_fmt(decls):
        print(line)


def _cli_data(parser, filenames=None, known=None):
    ArgumentParser = type(parser)
    common = ArgumentParser(add_help=False)
    # These flags will get processed by the top-level parse_args().
    add_verbosity_cli(common)
    add_traceback_cli(common)

    subs = parser.add_subparsers(dest='datacmd')

    sub = subs.add_parser('show', parents=[common])
    if known is None:
        sub.add_argument('--known', required=True)
    if filenames is None:
        sub.add_argument('filenames', metavar='FILE', nargs='+')

    sub = subs.add_parser('dump', parents=[common])
    if known is None:
        sub.add_argument('--known')
    sub.add_argument('--show', action='store_true')
    process_progress = add_progress_cli(sub)

    sub = subs.add_parser('check', parents=[common])
    if known is None:
        sub.add_argument('--known', required=True)

    def process_args(args, *, argv):
        if args.datacmd == 'dump':
            process_progress(args, argv)
    return process_args


def cmd_data(datacmd, filenames, known=None, *,
             _analyze=_analyze,
             formats=FORMATS,
             extracolumns=None,
             relroot=fsutil.USE_CWD,
             track_progress=None,
             **kwargs
             ):
    kwargs.pop('verbosity', None)
    usestdout = kwargs.pop('show', None)
    if datacmd == 'show':
        do_fmt = formats['summary']
        if isinstance(known, str):
            known, _ = _datafiles.get_known(known, extracolumns, relroot)
        for line in do_fmt(known):
            print(line)
    elif datacmd == 'dump':
        filenames, relroot = fsutil.fix_filenames(filenames, relroot=relroot)
        if track_progress:
            filenames = track_progress(filenames)
        analyzed = _analyze(filenames, **kwargs)
        analyzed.fix_filenames(relroot, normalize=False)
        if known is None or usestdout:
            outfile = io.StringIO()
            _datafiles.write_known(analyzed, outfile, extracolumns,
                                   relroot=relroot)
            print(outfile.getvalue())
        else:
            _datafiles.write_known(analyzed, known, extracolumns,
                                   relroot=relroot)
    elif datacmd == 'check':
        raise NotImplementedError(datacmd)
    else:
        raise ValueError(f'unsupported data command {datacmd!r}')


COMMANDS = {
    'check': (
        'analyze and fail if the given C source/header files have any problems',
        [_cli_check],
        cmd_check,
    ),
    'analyze': (
        'report on the state of the given C source/header files',
        [_cli_analyze],
        cmd_analyze,
    ),
    'data': (
        'check/manage local data (e.g. known types, ignored vars, caches)',
        [_cli_data],
        cmd_data,
    ),
}


#######################################
# the script

def parse_args(argv=sys.argv[1:], prog=sys.argv[0], *, subset=None):
    import argparse
    parser = argparse.ArgumentParser(
        prog=prog or get_prog(),
    )

    processors = add_commands_cli(
        parser,
        commands={k: v[1] for k, v in COMMANDS.items()},
        commonspecs=[
            add_verbosity_cli,
            add_traceback_cli,
        ],
        subset=subset,
    )

    args = parser.parse_args(argv)
    ns = vars(args)

    cmd = ns.pop('cmd')

    verbosity, traceback_cm = process_args_by_key(
        args,
        argv,
        processors[cmd],
        ['verbosity', 'traceback_cm'],
    )
    # "verbosity" is sent to the commands, so we put it back.
    args.verbosity = verbosity

    return cmd, ns, verbosity, traceback_cm


def main(cmd, cmd_kwargs):
    try:
        run_cmd = COMMANDS[cmd][0]
    except KeyError:
        raise ValueError(f'unsupported cmd {cmd!r}')
    run_cmd(**cmd_kwargs)


if __name__ == '__main__':
    cmd, cmd_kwargs, verbosity, traceback_cm = parse_args()
    configure_logger(verbosity)
    with traceback_cm:
        main(cmd, cmd_kwargs)

### File: analyze.py

In [ ]:
from c_parser.info import (
    KIND,
    TypeDeclaration,
    POTSType,
    FuncPtr,
)
from c_parser.match import (
    is_pots,
    is_funcptr,
)
from .info import (
    IGNORED,
    UNKNOWN,
    SystemType,
)
from .match import (
    is_system_type,
)


def get_typespecs(typedecls):
    typespecs = {}
    for decl in typedecls:
        if decl.shortkey not in typespecs:
            typespecs[decl.shortkey] = [decl]
        else:
            typespecs[decl.shortkey].append(decl)
    return typespecs


def analyze_decl(decl, typespecs, knowntypespecs, types, knowntypes, *,
                 analyze_resolved=None):
    resolved = resolve_decl(decl, typespecs, knowntypespecs, types)
    if resolved is None:
        # The decl is supposed to be skipped or ignored.
        return None
    if analyze_resolved is None:
        return resolved, None
    return analyze_resolved(resolved, decl, types, knowntypes)

# This alias helps us avoid name collisions.
_analyze_decl = analyze_decl


def analyze_type_decls(types, analyze_decl, handle_unresolved=True):
    unresolved = set(types)
    while unresolved:
        updated = []
        for decl in unresolved:
            resolved = analyze_decl(decl)
            if resolved is None:
                # The decl should be skipped or ignored.
                types[decl] = IGNORED
                updated.append(decl)
                continue
            typedeps, _ = resolved
            if typedeps is None:
                raise NotImplementedError(decl)
            if UNKNOWN in typedeps:
                # At least one dependency is unknown, so this decl
                # is not resolvable.
                types[decl] = UNKNOWN
                updated.append(decl)
                continue
            if None in typedeps:
                # XXX
                # Handle direct recursive types first.
                nonrecursive = 1
                if decl.kind is KIND.STRUCT or decl.kind is KIND.UNION:
                    nonrecursive = 0
                    i = 0
                    for member, dep in zip(decl.members, typedeps):
                        if dep is None:
                            if member.vartype.typespec != decl.shortkey:
                                nonrecursive += 1
                            else:
                                typedeps[i] = decl
                        i += 1
                if nonrecursive:
                    # We don't have all dependencies resolved yet.
                    continue
            types[decl] = resolved
            updated.append(decl)
        if updated:
            for decl in updated:
                unresolved.remove(decl)
        else:
            # XXX
            # Handle indirect recursive types.
            ...
            # We couldn't resolve the rest.
            # Let the caller deal with it!
            break
    if unresolved and handle_unresolved:
        if handle_unresolved is True:
            handle_unresolved = _handle_unresolved
        handle_unresolved(unresolved, types, analyze_decl)


def resolve_decl(decl, typespecs, knowntypespecs, types):
    if decl.kind is KIND.ENUM:
        typedeps = []
    else:
        if decl.kind is KIND.VARIABLE:
            vartypes = [decl.vartype]
        elif decl.kind is KIND.FUNCTION:
            vartypes = [decl.signature.returntype]
        elif decl.kind is KIND.TYPEDEF:
            vartypes = [decl.vartype]
        elif decl.kind is KIND.STRUCT or decl.kind is KIND.UNION:
            vartypes = [m.vartype for m in decl.members]
        else:
            # Skip this one!
            return None

        typedeps = []
        for vartype in vartypes:
            typespec = vartype.typespec
            if is_pots(typespec):
                typedecl = POTSType(typespec)
            elif is_system_type(typespec):
                typedecl = SystemType(typespec)
            elif is_funcptr(vartype):
                typedecl = FuncPtr(vartype)
            else:
                typedecl = find_typedecl(decl, typespec, typespecs)
                if typedecl is None:
                    typedecl = find_typedecl(decl, typespec, knowntypespecs)
                elif not isinstance(typedecl, TypeDeclaration):
                    raise NotImplementedError(repr(typedecl))
                if typedecl is None:
                    # We couldn't find it!
                    typedecl = UNKNOWN
                elif typedecl not in types:
                    # XXX How can this happen?
                    typedecl = UNKNOWN
                elif types[typedecl] is UNKNOWN:
                    typedecl = UNKNOWN
                elif types[typedecl] is IGNORED:
                    # We don't care if it didn't resolve.
                    pass
                elif types[typedecl] is None:
                    # The typedecl for the typespec hasn't been resolved yet.
                    typedecl = None
            typedeps.append(typedecl)
    return typedeps


def find_typedecl(decl, typespec, typespecs):
    specdecls = typespecs.get(typespec)
    if not specdecls:
        return None

    filename = decl.filename

    if len(specdecls) == 1:
        typedecl, = specdecls
        if '-' in typespec and typedecl.filename != filename:
            # Inlined types are always in the same file.
            return None
        return typedecl

    # Decide which one to return.
    candidates = []
    samefile = None
    for typedecl in specdecls:
        type_filename = typedecl.filename
        if type_filename == filename:
            if samefile is not None:
                # We expect type names to be unique in a file.
                raise NotImplementedError((decl, samefile, typedecl))
            samefile = typedecl
        elif filename.endswith('.c') and not type_filename.endswith('.h'):
            # If the decl is in a source file then we expect the
            # type to be in the same file or in a header file.
            continue
        candidates.append(typedecl)
    if not candidates:
        return None
    elif len(candidates) == 1:
        winner, = candidates
        # XXX Check for inline?
    elif '-' in typespec:
        # Inlined types are always in the same file.
        winner = samefile
    elif samefile is not None:
        # Favor types in the same file.
        winner = samefile
    else:
        # We don't know which to return.
        raise NotImplementedError((decl, candidates))

    return winner


#############################
# handling unresolved decls

class Skipped(TypeDeclaration):
    def __init__(self):
        _file = _name = _data = _parent = None
        super().__init__(_file, _name, _data, _parent, _shortkey='<skipped>')
_SKIPPED = Skipped()
del Skipped


def _handle_unresolved(unresolved, types, analyze_decl):
    #raise NotImplementedError(unresolved)

    dump = True
    dump = False
    if dump:
        print()
    for decl in types:  # Preserve the original order.
        if decl not in unresolved:
            assert types[decl] is not None, decl
            if types[decl] in (UNKNOWN, IGNORED):
                unresolved.add(decl)
                if dump:
                    _dump_unresolved(decl, types, analyze_decl)
                    print()
            else:
                assert types[decl][0] is not None, (decl, types[decl])
                assert None not in types[decl][0], (decl, types[decl])
        else:
            assert types[decl] is None
            if dump:
                _dump_unresolved(decl, types, analyze_decl)
                print()
    #raise NotImplementedError

    for decl in unresolved:
        types[decl] = ([_SKIPPED], None)

    for decl in types:
        assert types[decl]


def _dump_unresolved(decl, types, analyze_decl):
    if isinstance(decl, str):
        typespec = decl
        decl, = (d for d in types if d.shortkey == typespec)
    elif type(decl) is tuple:
        filename, typespec = decl
        if '-' in typespec:
            found = [d for d in types
                     if d.shortkey == typespec and d.filename == filename]
            #if not found:
            #    raise NotImplementedError(decl)
            decl, = found
        else:
            found = [d for d in types if d.shortkey == typespec]
            if not found:
                print(f'*** {typespec} ???')
                return
                #raise NotImplementedError(decl)
            else:
                decl, = found
    resolved = analyze_decl(decl)
    if resolved:
        typedeps, _ = resolved or (None, None)

    if decl.kind is KIND.STRUCT or decl.kind is KIND.UNION:
        print(f'*** {decl.shortkey} {decl.filename}')
        for member, mtype in zip(decl.members, typedeps):
            typespec = member.vartype.typespec
            if typespec == decl.shortkey:
                print(f'     ~~~~: {typespec:20} - {member!r}')
                continue
            status = None
            if is_pots(typespec):
                mtype = typespec
                status = 'okay'
            elif is_system_type(typespec):
                mtype = typespec
                status = 'okay'
            elif mtype is None:
                if '-' in member.vartype.typespec:
                    mtype, = [d for d in types
                              if d.shortkey == member.vartype.typespec
                              and d.filename == decl.filename]
                else:
                    found = [d for d in types
                             if d.shortkey == typespec]
                    if not found:
                        print(f' ???: {typespec:20}')
                        continue
                    mtype, = found
            if status is None:
                status = 'okay' if types.get(mtype) else 'oops'
            if mtype is _SKIPPED:
                status = 'okay'
                mtype = '<skipped>'
            elif isinstance(mtype, FuncPtr):
                status = 'okay'
                mtype = str(mtype.vartype)
            elif not isinstance(mtype, str):
                if hasattr(mtype, 'vartype'):
                    if is_funcptr(mtype.vartype):
                        status = 'okay'
                mtype = str(mtype).rpartition('(')[0].rstrip()
            status = '    okay' if status == 'okay' else f'--> {status}'
            print(f' {status}: {typespec:20} - {member!r} ({mtype})')
    else:
        print(f'*** {decl} ({decl.vartype!r})')
        if decl.vartype.typespec.startswith('struct ') or is_funcptr(decl):
            _dump_unresolved(
                (decl.filename, decl.vartype.typespec),
                types,
                analyze_decl,
            )

### File: datafiles.py

In [ ]:
import os.path

from c_common import fsutil
import c_common.tables as _tables
import c_parser.info as _info
import c_parser.match as _match
import c_parser.datafiles as _parser
from . import analyze as _analyze


#############################
# "known" decls

EXTRA_COLUMNS = [
    #'typedecl',
]


def get_known(known, extracolumns=None, *,
              analyze_resolved=None,
              handle_unresolved=True,
              relroot=fsutil.USE_CWD,
              ):
    if isinstance(known, str):
        known = read_known(known, extracolumns, relroot)
    return analyze_known(
        known,
        handle_unresolved=handle_unresolved,
        analyze_resolved=analyze_resolved,
    )


def read_known(infile, extracolumns=None, relroot=fsutil.USE_CWD):
    extracolumns = EXTRA_COLUMNS + (
        list(extracolumns) if extracolumns else []
    )
    known = {}
    for decl, extra in _parser.iter_decls_tsv(infile, extracolumns, relroot):
        known[decl] = extra
    return known


def analyze_known(known, *,
                  analyze_resolved=None,
                  handle_unresolved=True,
                  ):
    knowntypes = knowntypespecs = {}
    collated = _match.group_by_kinds(known)
    types = {decl: None for decl in collated['type']}
    typespecs = _analyze.get_typespecs(types)
    def analyze_decl(decl):
        return _analyze.analyze_decl(
            decl,
            typespecs,
            knowntypespecs,
            types,
            knowntypes,
            analyze_resolved=analyze_resolved,
        )
    _analyze.analyze_type_decls(types, analyze_decl, handle_unresolved)
    return types, typespecs


def write_known(rows, outfile, extracolumns=None, *,
                relroot=fsutil.USE_CWD,
                backup=True,
                ):
    extracolumns = EXTRA_COLUMNS + (
        list(extracolumns) if extracolumns else []
    )
    _parser.write_decls_tsv(
        rows,
        outfile,
        extracolumns,
        relroot=relroot,
        backup=backup,
    )


#############################
# ignored vars

IGNORED_COLUMNS = [
    'filename',
    'funcname',
    'name',
    'reason',
]
IGNORED_HEADER = '\t'.join(IGNORED_COLUMNS)


def read_ignored(infile, relroot=fsutil.USE_CWD):
    return dict(_iter_ignored(infile, relroot))


def _iter_ignored(infile, relroot):
    if relroot and relroot is not fsutil.USE_CWD:
        relroot = os.path.abspath(relroot)
    bogus = {_tables.EMPTY, _tables.UNKNOWN}
    for row in _tables.read_table(infile, IGNORED_HEADER, sep='\t'):
        *varidinfo, reason = row
        if _tables.EMPTY in varidinfo or _tables.UNKNOWN in varidinfo:
            varidinfo = tuple(None if v in bogus else v
                              for v in varidinfo)
        if reason in bogus:
            reason = None
        try:
            varid = _info.DeclID.from_row(varidinfo)
        except BaseException as e:
            e.add_note(f"Error occurred when processing row {varidinfo} in {infile}.")
            e.add_note(f"Could it be that you added a row which is not tab-delimited?")
            raise e
        varid = varid.fix_filename(relroot, formatted=False, fixroot=False)
        yield varid, reason


def write_ignored(variables, outfile, relroot=fsutil.USE_CWD):
    raise NotImplementedError
    if relroot and relroot is not fsutil.USE_CWD:
        relroot = os.path.abspath(relroot)
    reason = '???'
    #if not isinstance(varid, DeclID):
    #    varid = getattr(varid, 'parsed', varid).id
    decls = (d.fix_filename(relroot, fixroot=False) for d in decls)
    _tables.write_table(
        outfile,
        IGNORED_HEADER,
        sep='\t',
        rows=(r.render_rowdata() + (reason,) for r in decls),
    )

### File: info.py

In [ ]:
import os.path

from c_common import fsutil
from c_common.clsutil import classonly
import c_common.misc as _misc
from c_parser.info import (
    KIND,
    HighlevelParsedItem,
    Declaration,
    TypeDeclaration,
)
from c_parser.match import (
    is_type_decl,
)


IGNORED = _misc.Labeled('IGNORED')
UNKNOWN = _misc.Labeled('UNKNOWN')


class SystemType(TypeDeclaration):

    def __init__(self, name):
        super().__init__(None, name, None, None, _shortkey=name)


class Analyzed:
    _locked = False

    @classonly
    def is_target(cls, raw):
        if isinstance(raw, HighlevelParsedItem):
            return True
        else:
            return False

    @classonly
    def from_raw(cls, raw, **extra):
        if isinstance(raw, cls):
            if extra:
                # XXX ?
                raise NotImplementedError((raw, extra))
                #return cls(raw.item, raw.typedecl, **raw._extra, **extra)
            else:
                return info
        elif cls.is_target(raw):
            return cls(raw, **extra)
        else:
            raise NotImplementedError((raw, extra))

    @classonly
    def from_resolved(cls, item, resolved, **extra):
        if isinstance(resolved, TypeDeclaration):
            return cls(item, typedecl=resolved, **extra)
        else:
            typedeps, extra = cls._parse_raw_resolved(item, resolved, extra)
            if item.kind is KIND.ENUM:
                if typedeps:
                    raise NotImplementedError((item, resolved, extra))
            elif not typedeps:
                raise NotImplementedError((item, resolved, extra))
            return cls(item, typedeps, **extra or {})

    @classonly
    def _parse_raw_resolved(cls, item, resolved, extra_extra):
        if resolved in (UNKNOWN, IGNORED):
            return resolved, None
        try:
            typedeps, extra = resolved
        except (TypeError, ValueError):
            typedeps = extra = None
        if extra:
            # The resolved data takes precedence.
            extra = dict(extra_extra, **extra)
        if isinstance(typedeps, TypeDeclaration):
            return typedeps, extra
        elif typedeps in (None, UNKNOWN):
            # It is still effectively unresolved.
            return UNKNOWN, extra
        elif None in typedeps or UNKNOWN in typedeps:
            # It is still effectively unresolved.
            return typedeps, extra
        elif any(not isinstance(td, TypeDeclaration) for td in typedeps):
            raise NotImplementedError((item, typedeps, extra))
        return typedeps, extra

    def __init__(self, item, typedecl=None, **extra):
        assert item is not None
        self.item = item
        if typedecl in (UNKNOWN, IGNORED):
            pass
        elif item.kind is KIND.STRUCT or item.kind is KIND.UNION:
            if isinstance(typedecl, TypeDeclaration):
                raise NotImplementedError(item, typedecl)
            elif typedecl is None:
                typedecl = UNKNOWN
            else:
                typedecl = [UNKNOWN if d is None else d for d in typedecl]
        elif typedecl is None:
            typedecl = UNKNOWN
        elif typedecl and not isinstance(typedecl, TypeDeclaration):
            # All the other decls have a single type decl.
            typedecl, = typedecl
            if typedecl is None:
                typedecl = UNKNOWN
        self.typedecl = typedecl
        self._extra = extra
        self._locked = True

        self._validate()

    def _validate(self):
        item = self.item
        extra = self._extra
        # Check item.
        if not isinstance(item, HighlevelParsedItem):
            raise ValueError(f'"item" must be a high-level parsed item, got {item!r}')
        # Check extra.
        for key, value in extra.items():
            if key.startswith('_'):
                raise ValueError(f'extra items starting with {"_"!r} not allowed, got {extra!r}')
            if hasattr(item, key) and not callable(getattr(item, key)):
                raise ValueError(f'extra cannot override item, got {value!r} for key {key!r}')

    def __repr__(self):
        kwargs = [
            f'item={self.item!r}',
            f'typedecl={self.typedecl!r}',
            *(f'{k}={v!r}' for k, v in self._extra.items())
        ]
        return f'{type(self).__name__}({", ".join(kwargs)})'

    def __str__(self):
        try:
            return self._str
        except AttributeError:
            self._str, = self.render('line')
            return self._str

    def __hash__(self):
        return hash(self.item)

    def __eq__(self, other):
        if isinstance(other, Analyzed):
            return self.item == other.item
        elif isinstance(other, HighlevelParsedItem):
            return self.item == other
        elif type(other) is tuple:
            return self.item == other
        else:
            return NotImplemented

    def __gt__(self, other):
        if isinstance(other, Analyzed):
            return self.item > other.item
        elif isinstance(other, HighlevelParsedItem):
            return self.item > other
        elif type(other) is tuple:
            return self.item > other
        else:
            return NotImplemented

    def __dir__(self):
        names = set(super().__dir__())
        names.update(self._extra)
        names.remove('_locked')
        return sorted(names)

    def __getattr__(self, name):
        if name.startswith('_'):
            raise AttributeError(name)
        # The item takes precedence over the extra data (except if callable).
        try:
            value = getattr(self.item, name)
            if callable(value):
                raise AttributeError(name)
        except AttributeError:
            try:
                value = self._extra[name]
            except KeyError:
                pass
            else:
                # Speed things up the next time.
                self.__dict__[name] = value
                return value
            raise  # re-raise
        else:
            return value

    def __setattr__(self, name, value):
        if self._locked and name != '_str':
            raise AttributeError(f'readonly ({name})')
        super().__setattr__(name, value)

    def __delattr__(self, name):
        if self._locked:
            raise AttributeError(f'readonly ({name})')
        super().__delattr__(name)

    @property
    def decl(self):
        if not isinstance(self.item, Declaration):
            raise AttributeError('decl')
        return self.item

    @property
    def signature(self):
        # XXX vartype...
        ...

    @property
    def istype(self):
        return is_type_decl(self.item.kind)

    @property
    def is_known(self):
        if self.typedecl in (UNKNOWN, IGNORED):
            return False
        elif isinstance(self.typedecl, TypeDeclaration):
            return True
        else:
            return UNKNOWN not in self.typedecl

    def fix_filename(self, relroot=fsutil.USE_CWD, **kwargs):
        self.item.fix_filename(relroot, **kwargs)
        return self

    def as_rowdata(self, columns=None):
        # XXX finish!
        return self.item.as_rowdata(columns)

    def render_rowdata(self, columns=None):
        # XXX finish!
        return self.item.render_rowdata(columns)

    def render(self, fmt='line', *, itemonly=False):
        if fmt == 'raw':
            yield repr(self)
            return
        rendered = self.item.render(fmt)
        if itemonly or not self._extra:
            yield from rendered
            return
        extra = self._render_extra(fmt)
        if not extra:
            yield from rendered
        elif fmt in ('brief', 'line'):
            rendered, = rendered
            extra, = extra
            yield f'{rendered}\t{extra}'
        elif fmt == 'summary':
            raise NotImplementedError(fmt)
        elif fmt == 'full':
            yield from rendered
            for line in extra:
                yield f'\t{line}'
        else:
            raise NotImplementedError(fmt)

    def _render_extra(self, fmt):
        if fmt in ('brief', 'line'):
            yield str(self._extra)
        else:
            raise NotImplementedError(fmt)


class Analysis:

    _item_class = Analyzed

    @classonly
    def build_item(cls, info, resolved=None, **extra):
        if resolved is None:
            return cls._item_class.from_raw(info, **extra)
        else:
            return cls._item_class.from_resolved(info, resolved, **extra)

    @classmethod
    def from_results(cls, results):
        self = cls()
        for info, resolved in results:
            self._add_result(info, resolved)
        return self

    def __init__(self, items=None):
        self._analyzed = {type(self).build_item(item): None
                          for item in items or ()}

    def __repr__(self):
        return f'{type(self).__name__}({list(self._analyzed.keys())})'

    def __iter__(self):
        #yield from self.types
        #yield from self.functions
        #yield from self.variables
        yield from self._analyzed

    def __len__(self):
        return len(self._analyzed)

    def __getitem__(self, key):
        if type(key) is int:
            for i, val in enumerate(self._analyzed):
                if i == key:
                    return val
            else:
                raise IndexError(key)
        else:
            return self._analyzed[key]

    def fix_filenames(self, relroot=fsutil.USE_CWD, **kwargs):
        if relroot and relroot is not fsutil.USE_CWD:
            relroot = os.path.abspath(relroot)
        for item in self._analyzed:
            item.fix_filename(relroot, fixroot=False, **kwargs)

    def _add_result(self, info, resolved):
        analyzed = type(self).build_item(info, resolved)
        self._analyzed[analyzed] = None
        return analyzed

### File: match.py

In [ ]:
import os.path

from c_parser import (
    info as _info,
    match as _match,
)


_KIND = _info.KIND


# XXX Use known.tsv for these?
SYSTEM_TYPES = {
    'int8_t',
    'uint8_t',
    'int16_t',
    'uint16_t',
    'int32_t',
    'uint32_t',
    'int64_t',
    'uint64_t',
    'size_t',
    'ssize_t',
    'intptr_t',
    'uintptr_t',
    'wchar_t',
    '',
    # OS-specific
    'pthread_cond_t',
    'pthread_mutex_t',
    'pthread_key_t',
    'atomic_int',
    'atomic_uintptr_t',
    '',
    # lib-specific
    'WINDOW',  # curses
    'XML_LChar',
    'XML_Size',
    'XML_Parser',
    'enum XML_Error',
    'enum XML_Status',
    '',
}


def is_system_type(typespec):
    return typespec in SYSTEM_TYPES


##################################
# decl matchers

def is_public(decl):
    if not decl.filename.endswith('.h'):
        return False
    if 'Include' not in decl.filename.split(os.path.sep):
        return False
    return True


def is_process_global(vardecl):
    kind, storage, _, _, _ = _info.get_parsed_vartype(vardecl)
    if kind is not _KIND.VARIABLE:
        raise NotImplementedError(vardecl)
    if 'static' in (storage or ''):
        return True

    if hasattr(vardecl, 'parent'):
        parent = vardecl.parent
    else:
        parent = vardecl.get('parent')
    return not parent


def is_fixed_type(vardecl):
    if not vardecl:
        return None
    _, _, _, typespec, abstract = _info.get_parsed_vartype(vardecl)
    if 'typeof' in typespec:
        raise NotImplementedError(vardecl)
    elif not abstract:
        return True

    if '*' not in abstract:
        # XXX What about []?
        return True
    elif _match._is_funcptr(abstract):
        return True
    else:
        for after in abstract.split('*')[1:]:
            if not after.lstrip().startswith('const'):
                return False
        else:
            return True


def is_immutable(vardecl):
    if not vardecl:
        return None
    if not is_fixed_type(vardecl):
        return False
    _, _, typequal, _, _ = _info.get_parsed_vartype(vardecl)
    # If there, it can only be "const" or "volatile".
    return typequal == 'const'


def is_public_api(decl):
    if not is_public(decl):
        return False
    if decl.kind is _KIND.TYPEDEF:
        return True
    elif _match.is_type_decl(decl):
        return not _match.is_forward_decl(decl)
    else:
        return _match.is_external_reference(decl)


def is_public_declaration(decl):
    if not is_public(decl):
        return False
    if decl.kind is _KIND.TYPEDEF:
        return True
    elif _match.is_type_decl(decl):
        return _match.is_forward_decl(decl)
    else:
        return _match.is_external_reference(decl)


def is_public_definition(decl):
    if not is_public(decl):
        return False
    if decl.kind is _KIND.TYPEDEF:
        return True
    elif _match.is_type_decl(decl):
        return not _match.is_forward_decl(decl)
    else:
        return not _match.is_external_reference(decl)


def is_public_impl(decl):
    if not _KIND.is_decl(decl.kind):
        return False
    # See filter_forward() about "is_public".
    return getattr(decl, 'is_public', False)


def is_module_global_decl(decl):
    if is_public_impl(decl):
        return False
    if _match.is_forward_decl(decl):
        return False
    return not _match.is_local_var(decl)


##################################
# filtering with matchers

def filter_forward(items, *, markpublic=False):
    if markpublic:
        public = set()
        actual = []
        for item in items:
            if is_public_api(item):
                public.add(item.id)
            elif not _match.is_forward_decl(item):
                actual.append(item)
            else:
                # non-public duplicate!
                # XXX
                raise Exception(item)
        for item in actual:
            _info.set_flag(item, 'is_public', item.id in public)
            yield item
    else:
        for item in items:
            if _match.is_forward_decl(item):
                continue
            yield item


##################################
# grouping with matchers

def group_by_storage(decls, **kwargs):
    def is_module_global(decl):
        if not is_module_global_decl(decl):
            return False
        if decl.kind == _KIND.VARIABLE:
            if _info.get_effective_storage(decl) == 'static':
                # This is covered by is_static_module_global().
                return False
        return True
    def is_static_module_global(decl):
        if not _match.is_global_var(decl):
            return False
        return _info.get_effective_storage(decl) == 'static'
    def is_static_local(decl):
        if not _match.is_local_var(decl):
            return False
        return _info.get_effective_storage(decl) == 'static'
    #def is_local(decl):
    #    if not _match.is_local_var(decl):
    #        return False
    #    return _info.get_effective_storage(decl) != 'static'
    categories = {
        #'extern': is_extern,
        'published': is_public_impl,
        'module-global': is_module_global,
        'static-module-global': is_static_module_global,
        'static-local': is_static_local,
    }
    return _match.group_by_category(decls, categories, **kwargs)